# STEP 1: Setup and Installation

In [1]:
# Install ultralytics (YOLOv11)
!pip install ultralytics

print("✓ Libraries installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.6 MB/s eta 0:00:00
✓ Libraries installed successfully!


In [2]:
# Import libraries
import pandas as pd
print(pd.__version__)
import os
import shutil
import yaml
from pathlib import Path
import sys

from ultralytics import YOLO

import cv2

2.2.2
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# STEP 2: Dataset Preparation

In [3]:
SCRIPT_DIR = Path('/kaggle/working')  # Kaggle's writable directory
WORK_DIR = SCRIPT_DIR
DATASET_PATH = Path('/kaggle/input/datasets/lylmsc/wider-face-for-yolo-training')

if not DATASET_PATH.exists():
    print(f"Dataset not found at {DATASET_PATH}. Please ensure the 'dataset' folder is here.")
    sys.exit(1)
print(f"Using dataset at: {DATASET_PATH}")

Using dataset at: /kaggle/input/datasets/lylmsc/wider-face-for-yolo-training


# STEP 3: Convert CSV Annotations to YOLO Format

In [4]:
def prepare_wider_dataset():
    """
    Organize the WIDER Face dataset for YOLO training.
    The dataset is already in YOLO format, so we just need to split it into train/val/test sets.
    """
    # Create directory structure in /kaggle/working
    dirs = ['images/train', 'images/val', 'images/test',
            'labels/train', 'labels/val', 'labels/test']
    for d in dirs:
        os.makedirs(f'{WORK_DIR}/{d}', exist_ok=True)

    # Source directories
    source_images = DATASET_PATH / 'images'
    source_labels = DATASET_PATH / 'labels'

    if not source_images.exists() or not source_labels.exists():
        print(f"⚠️ Error: Expected 'images' and 'labels' folders in {DATASET_PATH}")
        sys.exit(1)

    # Get all image files
    image_files = sorted(list(source_images.glob('*.jpg')))
    if len(image_files) == 0:
        image_files = sorted(list(source_images.glob('*.png')))
    
    print(f"\nFound {len(image_files)} images in the dataset")

    # Split dataset: 80% train, 10% val, 10% test
    total = len(image_files)
    train_split = int(total * 0.8)
    val_split = int(total * 0.9)

    splits = {
        'train': image_files[:train_split],
        'val': image_files[train_split:val_split],
        'test': image_files[val_split:]
    }

    # Copy files to respective directories
    for split_name, files in splits.items():
        print(f"\nProcessing {split_name} split ({len(files)} images)...")
        
        processed_images = 0
        processed_labels = 0
        missing_labels = 0

        for img_path in files:
            # Copy image
            dst_img = WORK_DIR / 'images' / split_name / img_path.name
            shutil.copy(img_path, dst_img)
            processed_images += 1

            # Copy corresponding label file
            label_name = img_path.stem + '.txt'
            src_label = source_labels / label_name
            dst_label = WORK_DIR / 'labels' / split_name / label_name

            if src_label.exists():
                shutil.copy(src_label, dst_label)
                processed_labels += 1
            else:
                # Create empty label file if no annotations
                dst_label.touch()
                missing_labels += 1

        print(f"  ✓ Copied {processed_images} images")
        print(f"  ✓ Copied {processed_labels} labels")
        if missing_labels > 0:
            print(f"  ⚠️ Created {missing_labels} empty label files (no faces in these images)")

    print("\n" + "="*60)
    print("WIDER Face dataset preparation complete!")
    print("="*60)

# Run dataset preparation
prepare_wider_dataset()



Found 12880 images in the dataset

Processing train split (10304 images)...
  ✓ Copied 10304 images
  ✓ Copied 10304 labels

Processing val split (1288 images)...
  ✓ Copied 1288 images
  ✓ Copied 1288 labels

Processing test split (1288 images)...
  ✓ Copied 1288 images
  ✓ Copied 1288 labels

WIDER Face dataset preparation complete!


# STEP 5: Create data.yaml Configuration File

In [5]:
data_yaml = {
    'path': str(WORK_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 1,  # number of classes
    'names': ['face']  # class names
}

with open(f'{WORK_DIR}/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("\ndata.yaml created:")
print(yaml.dump(data_yaml, default_flow_style=False))


data.yaml created:
names:
- face
nc: 1
path: /kaggle/working
test: images/test
train: images/train
val: images/val



# STEP 6: Verify Dataset

In [6]:
def verify_dataset():
    """Check dataset integrity"""
    train_images = len(os.listdir(f'{WORK_DIR}/images/train'))
    train_labels = len(os.listdir(f'{WORK_DIR}/labels/train'))
    val_images = len(os.listdir(f'{WORK_DIR}/images/val'))
    val_labels = len(os.listdir(f'{WORK_DIR}/labels/val'))
    test_images = len(os.listdir(f'{WORK_DIR}/images/test'))
    test_labels = len(os.listdir(f'{WORK_DIR}/labels/test'))

    print("\nDataset Summary:")
    print(f"Train - Images: {train_images}, Labels: {train_labels}")
    print(f"Val   - Images: {val_images}, Labels: {val_labels}")
    print(f"Test  - Images: {test_images}, Labels: {test_labels}")

    # Sample a label file to verify format
    if train_labels > 0:
        sample_label = os.listdir(f'{WORK_DIR}/labels/train')[0]
        print(f"\nSample label file ({sample_label}):")
        with open(f'{WORK_DIR}/labels/train/{sample_label}', 'r') as f:
            lines = f.readlines()[:3]  # Show first 3 annotations
            for line in lines:
                print(f"  {line.strip()}")
            if len(lines) > 3:
                print(f"  ... and more")

    if train_images == 0 or val_images == 0:
        print("\n⚠️ WARNING: No images found! Please check dataset organization.")
    elif train_images != train_labels or val_images != val_labels:
        print("\n⚠️ WARNING: Mismatch between images and labels!")
    else:
        print("\n✓ Dataset looks good!")

verify_dataset()


Dataset Summary:
Train - Images: 10304, Labels: 10304
Val   - Images: 1288, Labels: 1288
Test  - Images: 1288, Labels: 1288

Sample label file (wider_12516.txt):
  0 0.85546875 0.16398243045387995 0.0546875 0.10980966325036604
  0 0.3173828125 0.16398243045387995 0.0185546875 0.029282576866764276
  0 0.296875 0.21815519765739386 0.044921875 0.07906295754026355

✓ Dataset looks good!


# STEP 7: Initialize YOLOv11 Model

In [7]:
# Load a pretrained YOLOv11 model (nano, small, medium, large, or extra-large)
model = YOLO('yolo11n.pt')  # 'n' for nano (fastest), 's', 'm', 'l', 'x' for larger models

print("\nModel loaded successfully!")


Model loaded successfully!


# Save checkpoints

In [8]:
import json
import subprocess
from kaggle_secrets import UserSecretsClient

KAGGLE_USERNAME = "a21101131"
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")  # ← secret LABEL, not username

# Write kaggle.json so the CLI works
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

CHECKPOINT_DATASET  = "yolo-checkpoints"
CHECKPOINT_SAVE_DIR = Path('/kaggle/working/checkpoint_upload')
EXPERIMENT_NAME     = 'yolov11_face_improved'   # ← defined here so callback can see it
SAVE_EVERY_N_EPOCHS = 10                         # push every N epochs; lower = safer but slower

def push_checkpoint_to_dataset():
    """Copy last.pt to Kaggle dataset storage. Creates the dataset on first push,
    versions it on later pushes. Safe to call at any time."""
    CHECKPOINT_SAVE_DIR.mkdir(exist_ok=True)
    src = WORK_DIR / EXPERIMENT_NAME / 'weights' / 'last.pt'
    if not src.exists():
        print(f"⚠️  Checkpoint not found at {src}")
        return
    shutil.copy(src, CHECKPOINT_SAVE_DIR / 'last.pt')

    meta = {
        "title": CHECKPOINT_DATASET,
        "id": f"{KAGGLE_USERNAME}/{CHECKPOINT_DATASET}",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(CHECKPOINT_SAVE_DIR / 'dataset-metadata.json', 'w') as f:
        json.dump(meta, f)

    # Does the dataset already exist?
    dataset_full_id = f"{KAGGLE_USERNAME}/{CHECKPOINT_DATASET}"
    status = subprocess.run(['kaggle', 'datasets', 'status', dataset_full_id],
                            capture_output=True, text=True)
    exists = 'ready' in status.stdout.lower()

    epoch_msg = getattr(push_checkpoint_to_dataset, "_epoch", "?")
    if exists:
        cmd = ['kaggle', 'datasets', 'version', '-p', str(CHECKPOINT_SAVE_DIR),
               '-m', f'auto checkpoint epoch {epoch_msg}', '--dir-mode', 'zip']
    else:
        print("ℹ️  Dataset doesn't exist yet — creating it...")
        cmd = ['kaggle', 'datasets', 'create', '-p', str(CHECKPOINT_SAVE_DIR),
               '--dir-mode', 'zip']

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✓ Checkpoint pushed to Kaggle dataset (epoch {epoch_msg})!")
    else:
        print(f"⚠️  Push failed:\n{result.stderr}")

def save_checkpoint_callback(trainer):
    """Registered on on_train_epoch_end — fires automatically each epoch."""
    epoch = trainer.epoch + 1
    push_checkpoint_to_dataset._epoch = epoch   # store for the log message
    if epoch % SAVE_EVERY_N_EPOCHS == 0:
        print(f"\n[Checkpoint] Epoch {epoch} — pushing weights to Kaggle...")
        push_checkpoint_to_dataset()

print(f"✓ Checkpoint helpers ready. Will push every {SAVE_EVERY_N_EPOCHS} epochs automatically.")

✓ Checkpoint helpers ready. Will push every 10 epochs automatically.


# STEP 8: Train the Model

In [9]:
import os
print(os.listdir('/kaggle/input/models/a21101131/checkpoint2/other/default/1'))

['last (1).pt']


In [10]:
DATA_YAML       = WORK_DIR / 'data.yaml'
CHECKPOINT_PATH = Path(f'/kaggle/input/models/a21101131/checkpoint2/other/default/1/last (1).pt')

if CHECKPOINT_PATH.exists():
    print(f"[INFO] Resuming from checkpoint: {CHECKPOINT_PATH}")
    model = YOLO(str(CHECKPOINT_PATH))
    model.add_callback("on_train_epoch_end", save_checkpoint_callback)
    results = model.train(resume=True)
else:
    print("[INFO] No checkpoint found. Starting fresh...")
    model = YOLO('yolo11n.pt')
    model.add_callback("on_train_epoch_end", save_checkpoint_callback)
    results = model.train(
        data=str(DATA_YAML),
        epochs=100,
        patience=50,
        imgsz=1280,
        batch=8,
        copy_paste=0.3,
        mixup=0.15,
        erasing=0.4,
        close_mosaic=30,
        device=0,
        workers=2,
        project=str(WORK_DIR),
        name=EXPERIMENT_NAME,
        save=True,
        plots=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=0.0,
        translate=0.1,
        scale=0.5,
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
    )

# Final push — runs if training completes normally before timeout
print("\n[Checkpoint] Training done — pushing final weights...")
push_checkpoint_to_dataset()
print("✓ Final checkpoint saved.")


[INFO] Resuming from checkpoint: /kaggle/input/models/a21101131/checkpoint2/other/default/1/last (1).pt
Ultralytics 8.4.70 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=30, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=/kaggle/input/models/a21101131/checkpoint2/other/default/1/last (1

In [11]:
# The manual checkpoint cell below is no longer needed.
# Checkpoints are now pushed automatically:
#   • Every 5 epochs via save_checkpoint_callback  (configure SAVE_EVERY_N_EPOCHS above)
#   • Once more automatically after training completes
print("ℹ️  Automatic checkpointing is active. No manual action needed.")


ℹ️  Automatic checkpointing is active. No manual action needed.


# STEP 9: Evaluate the Model

In [12]:
# Load the best model
best_model = YOLO(str(WORK_DIR / 'yolov11_face' / 'weights' / 'best.pt'))

# Validate
metrics = best_model.val()

print("\nValidation Metrics:")

# Use the correct attribute names from the metrics object
results_dict = metrics.results_dict

print(f"mAP50: {results_dict['metrics/mAP50(B)']:.4f}")
print(f"mAP50-95: {results_dict['metrics/mAP50-95(B)']:.4f}")
print(f"Precision: {results_dict['metrics/precision(B)']:.4f}")
print(f"Recall: {results_dict['metrics/recall(B)']:.4f}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/yolov11_face/weights/best.pt'

# STEP 10: Test on Sample Images

In [ ]:
import glob
from PIL import Image as PILImage

val_images = glob.glob(f'{WORK_DIR}/images/val/*.jpg')
if not val_images:
    val_images = glob.glob(f'{WORK_DIR}/images/val/*.png')
val_images = val_images[:5]  # Take first 5 images

print("\nRunning inference on sample images...")

os.makedirs(WORK_DIR / 'sample_outputs', exist_ok=True)
for img_path in val_images:
    results = best_model.predict(str(img_path))

    # Display or save results
    for r in results:
        im_array = r.plot()  # plot a BGR numpy array of predictions
        im = PILImage.fromarray(im_array[..., ::-1])  # RGB PIL image
        display(im)

# STEP 11: Export Model

In [ ]:
# Export to different formats
print("\nExporting model...")

# Export to ONNX (for deployment)
best_model.export(format='onnx')

# Export to TensorRT (for NVIDIA devices)
# best_model.export(format='engine')

# Export to TFLite (for mobile)
# best_model.export(format='tflite')

print("\n✓ Model exported successfully!")

# STEP 12: Save Model Locally

In [ ]:
# Copy best model to a local folder (same directory as script)
output_dir = WORK_DIR / 'saved_model'
output_dir.mkdir(parents=True, exist_ok=True)

weights_dir = WORK_DIR / 'yolov11_face' / 'weights'
for name in ('best.pt', 'last.pt'):
    src = weights_dir / name
    if src.exists():
        shutil.copy(src, output_dir / name)

print(f"\n✓ Models saved to: {output_dir}")

# Add this at the end of your notebook
shutil.copy(WORK_DIR / 'yolov11_face_improved' / 'weights' / 'last.pt', 
            WORK_DIR / 'last_checkpoint.pt')
print("✓ Download 'last_checkpoint.pt' from the Output panel on the right →")

# STEP 13: Video Processing for Face Detection

In [ ]:
def process_video_for_face_detection(
    video_path,
    model,
    output_dir='face_detection_output',
    frame_skip=1,
    conf_threshold=0.25,
    save_annotated_video=True,
    save_cropped_faces=True,
    save_frames=False
):
    """
    Process a video to detect faces in each frame.
    
    Parameters:
    -----------
    video_path : str
        Path to the input video file
    model : YOLO
        Trained YOLO model for face detection
    output_dir : str
        Directory to save outputs
    frame_skip : int
        Process every Nth frame (1 = all frames, 2 = every other frame, etc.)
    conf_threshold : float
        Confidence threshold for detections (0-1)
    save_annotated_video : bool
        Save video with bounding boxes drawn
    save_cropped_faces : bool
        Save individual cropped face images
    save_frames : bool
        Save all processed frames
    
    Returns:
    --------
    dict : Statistics about the processing
    """
    
    # Create output directories
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    if save_cropped_faces:
        faces_dir = output_path / 'cropped_faces'
        faces_dir.mkdir(exist_ok=True)
    
    if save_frames:
        frames_dir = output_path / 'frames'
        frames_dir.mkdir(exist_ok=True)
    
    # Open video
    cap = cv2.VideoCapture(str(video_path))
    
    if not cap.isOpened():
        raise ValueError(f"Could not open video file: {video_path}")
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"\n{'='*60}")
    print(f"Video Information:")
    print(f"  Resolution: {width}x{height}")
    print(f"  FPS: {fps}")
    print(f"  Total Frames: {total_frames}")
    print(f"  Duration: {total_frames/fps:.2f} seconds")
    print(f"{'='*60}\n")
    
    # Prepare video writer if saving annotated video
    if save_annotated_video:
        output_video_path = output_path / 'annotated_video.mp4'
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out_video = cv2.VideoWriter(
            str(output_video_path), 
            fourcc, 
            fps // frame_skip,
            (width, height)
        )
    
    # Statistics
    stats = {
        'total_frames_processed': 0,
        'total_faces_detected': 0,
        'frames_with_faces': 0,
        'face_count_per_frame': []
    }
    
    frame_count = 0
    face_id = 0
    
    print("Processing video...")
    
    while True:
        ret, frame = cap.read()
        
        if not ret:
            break
        
        # Skip frames if needed
        if frame_count % frame_skip != 0:
            frame_count += 1
            continue
        
        # Run face detection
        results = model.predict(
            frame, 
            conf=conf_threshold,
            verbose=False
        )
        
        # Process results
        num_faces = 0
        annotated_frame = frame.copy()
        
        for result in results:
            boxes = result.boxes
            
            for box in boxes:
                # Get bounding box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                conf = box.conf[0].cpu().numpy()
                
                # Draw bounding box on frame
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    annotated_frame, 
                    f'Face {conf:.2f}', 
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 
                    0.5, 
                    (0, 255, 0), 
                    2
                )
                
                # Crop and save face
                if save_cropped_faces:
                    face_crop = frame[y1:y2, x1:x2]
                    if face_crop.size > 0:
                        face_path = faces_dir / f'face_{frame_count:06d}_{face_id:04d}.jpg'
                        cv2.imwrite(str(face_path), face_crop)
                        face_id += 1
                
                num_faces += 1
        
        # Update statistics
        stats['total_frames_processed'] += 1
        stats['total_faces_detected'] += num_faces
        stats['face_count_per_frame'].append(num_faces)
        if num_faces > 0:
            stats['frames_with_faces'] += 1
        
        # Save annotated frame to video
        if save_annotated_video:
            out_video.write(annotated_frame)
        
        # Save individual frames if requested
        if save_frames:
            frame_path = frames_dir / f'frame_{frame_count:06d}.jpg'
            cv2.imwrite(str(frame_path), annotated_frame)
        
        # Progress indicator
        if frame_count % 30 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"Progress: {progress:.1f}% | Frame {frame_count}/{total_frames} | Faces: {num_faces}")
        
        frame_count += 1
    
    # Cleanup
    cap.release()
    if save_annotated_video:
        out_video.release()
    
    # Calculate final statistics
    stats['avg_faces_per_frame'] = stats['total_faces_detected'] / max(stats['total_frames_processed'], 1)
    stats['max_faces_in_frame'] = max(stats['face_count_per_frame']) if stats['face_count_per_frame'] else 0
    
    # Print summary
    print(f"\n{'='*60}")
    print("Processing Complete!")
    print(f"{'='*60}")
    print(f"Frames Processed: {stats['total_frames_processed']}")
    print(f"Total Faces Detected: {stats['total_faces_detected']}")
    print(f"Frames with Faces: {stats['frames_with_faces']}")
    print(f"Average Faces per Frame: {stats['avg_faces_per_frame']:.2f}")
    print(f"Max Faces in Single Frame: {stats['max_faces_in_frame']}")
    print(f"\nOutputs saved to: {output_path}")
    if save_annotated_video:
        print(f"  ✓ Annotated video: {output_video_path}")
    if save_cropped_faces:
        print(f"  ✓ Cropped faces: {faces_dir}/ ({face_id} faces)")
    if save_frames:
        print(f"  ✓ Individual frames: {frames_dir}/")
    print(f"{'='*60}\n")
    
    return stats

print("✓ Video processing function defined!")

# STEP 14: Process a Video File

In [ ]:
# Upload your video to Kaggle first, then specify the path
# Example: VIDEO_PATH = '/kaggle/input/your-video-dataset/video.mp4'

VIDEO_PATH = '/kaggle/input/your-video-dataset/your_video.mp4'  # UPDATE THIS PATH

# Load your trained model
best_model = YOLO(str(WORK_DIR / 'yolov11_face' / 'weights' / 'best.pt'))

# Process the video
stats = process_video_for_face_detection(
    video_path=VIDEO_PATH,
    model=best_model,
    output_dir=str(WORK_DIR / 'face_detection_output'),
    frame_skip=1,              # Process every frame
    conf_threshold=0.3,        # Confidence threshold
    save_annotated_video=True, # Save video with bounding boxes
    save_cropped_faces=True,   # Save individual face crops
    save_frames=False          # Don't save all frames (set True if needed)
)

print("\n✓ Video processing complete!")
print(f"\nStatistics: {stats}")

# STEP 15: Display Sample Detected Faces

In [ ]:
import glob
from IPython.display import display

# Get first 10 detected faces
face_images = sorted(glob.glob(str(WORK_DIR / 'face_detection_output/cropped_faces/*.jpg')))[:10]

print(f"Displaying {len(face_images)} sample faces:\n")

for face_path in face_images:
    img = Image.open(face_path)
    print(f"Face: {Path(face_path).name}")
    display(img)
    print("-" * 40)